In [1]:
import scipy.optimize as opt
import numpy as np
import xarray as xr
import pandas as pd
import os
#PyGEM
import pygem.pygem_input as pygem_prms
import pygem.oggm_compat as oggm
import pygem.pygem_modelsetup as modelsetup

In [ ]:
# READ GULKANA ELEVATION DATA FROM RGI 6.0
gulkana = '01.00570'
glac_no = [gulkana]

# ----- RGI DATA -----
# Filepath for RGI files
main_directory = os.getcwd()
rgi_fp = main_directory + '/../RGI/rgi60/00_rgi60_attribs/'
assert os.path.exists(rgi_fp), 'RGI filepath does not exist. PyGEM requires RGI data to run.'
# Column names
rgi_lat_colname = 'CenLat'
rgi_lon_colname = 'CenLon_360' # REQUIRED OTHERWISE GLACIERS IN WESTERN HEMISPHERE USE 0 deg
elev_colname = 'elev'
indexname = 'GlacNo'
rgi_O1Id_colname = 'glacno'
rgi_glacno_float_colname = 'RGIId_float'
# Column names from table to drop (list names or accept an empty list)
rgi_cols_drop = ['GLIMSId','BgnDate','EndDate','Status','Linkages','Name']

region = glac_no[0][0:2]
for i in os.listdir(rgi_fp):
    if i.startswith(str(region)) and i.endswith('.csv'):
        rgi_fn = i
glacier_table = pd.read_csv(rgi_fp + rgi_fn,skipinitialspace=True,index_col='RGIId')
glacier_table.drop(rgi_cols_drop, axis=1, inplace=True)
gulkana_tbl = glacier_table.loc['RGI60-'+glac_no[0]]
print('Gulkana Glacier Table:')
print(gulkana_tbl)
elev_points = gulkana_tbl[['Zmin','Zmed','Zmax']]

In [ ]:
#%% MODEL PROPERTIES
density_ice = 900           # Density of ice [kg m-3] (or Gt / 1000 km3)
density_water = 1000        # Density of water [kg m-3]
area_ocean = 362.5 * 1e12   # Area of ocean [m2] (Cogley, 2012 from Marzeion et al. 2020)
k_ice = 2.33                # Thermal conductivity of ice [J s-1 K-1 m-1] recall (W = J s-1)
k_air = 0.023               # Thermal conductivity of air [J s-1 K-1 m-1] (Mellor, 1997)
k_air = 0.001               # Thermal conductivity of air [J s-1 K-1 m-1]
ch_ice = 1890000            # Volumetric heat capacity of ice [J K-1 m-3] (density=900, heat_capacity=2100 J K-1 kg-1)
ch_air = 1297               # Volumetric Heat capacity of air [J K-1 m-3] (density=1.29, heat_capacity=1005 J K-1 kg-1)
Lh_rf = 333550              # Latent heat of fusion [J kg-1]
tolerance = 1e-12           # Model tolerance (used to remove low values caused by rounding errors)
gravity = 9.81              # Gravity [m s-2]
pressure_std = 101325       # Standard pressure [Pa]
temp_std = 288.15           # Standard temperature [K]
R_gas = 8.3144598           # Universal gas constant [J mol-1 K-1]
molarmass_air = 0.0289644   # Molar mass of Earth's air [kg mol-1]

In [5]:
#READ GULKANA ELEVATIONS FROM OGGM GDIRS
gdir = oggm.single_flowline_glacier_directory(glac_no[0], logging_level='CRITICAL')
fls = oggm.get_glacier_zwh(gdir)
#filter out zero bins to get only initial glacier volume
fls = fls.iloc[np.nonzero(fls['h'].to_numpy())]
z_stats = np.array([np.min(fls['z']),np.median(fls['z']),np.max(fls['z'])])

print('Min, median and max bin elevations from OGGM flowlines:')
print(pd.Series(z_stats,index=['Zmin','Zmed','Zmax']))
print('Min, median and max elevation from RGI')
print(elev_points)

median_index = np.where(fls['z']==z_stats[1])[0][0]
w_stats = np.array([fls['w'][52],fls['w'][median_index],fls['w'][0]])
h_stats = np.array([fls['h'][52],fls['h'][median_index],fls['h'][0]])
geo_index = ['Bottom','Middle','Top']
geometry = pd.DataFrame({'z':z_stats,'w':w_stats,'h':h_stats},index=geo_index)

Min, median and max bin elevations from OGGM flowlines:
Zmin    1172.809436
Zmed    1628.401286
Zmax    2330.675374
dtype: float64
Min, median and max elevation from RGI
Zmin    1162.0
Zmed    1858.0
Zmax    2438.0
Name: RGI60-01.00570, dtype: float64


In [6]:
print(geometry)

                  z            w           h
Bottom  1172.809436   168.584994  123.288910
Middle  1628.401286  1680.294979  136.998843
Top     2330.675374   649.658441   60.315266


In [4]:
varname = 'precip'
hourlyfp = '~/research/climate_data/ERA5/ERA5_hourly/ERA5_varname_hourly'.replace('varname',varname)
da_0 = xr.open_dataarray(hourlyfp + '_80_89.nc')
da_1 = xr.open_dataarray(hourlyfp + '_90_99.nc')
da_2 = xr.open_dataarray(hourlyfp + '_00_09.nc')
da_3 = xr.open_dataarray(hourlyfp + '_10_22.nc')
print(len(da_0)+len(da_1)+len(da_2)+len(da_3))
da = xr.merge([da_0,da_1,da_2,da_3])
da.to_netcdf(hourlyfp+'.nc')

376240


In [11]:
xr.open_dataarray('~/research/climate_data/ERA5/ERA5_hourly/ERA5_temp_h.nc')

<xarray.DataArray 't2m' (time: 376221, latitude: 1, longitude: 1, expver: 2)>
[752442 values with dtype=float32]
Coordinates:
  * time       (time) datetime64[ns] 1980-01-01 ... 2022-12-01T20:00:00
  * longitude  (longitude) float32 -142.5
  * latitude   (latitude) float32 63.2
  * expver     (expver) int32 5 1
Attributes:
    units:      K
    long_name:  2 metre temperature

In [9]:
import class_climate
import pygem.pygem_modelsetup as modelsetup
gcm = class_climate.GCM(name='ERA5-daily')
main_glac_rgi = modelsetup.selectglaciersrgitable(glac_no=pygem_prms.glac_no)
dates_table = modelsetup.datesmodelrun(startyear=2000, endyear=2019, spinupyears=0, 
                                           option_wateryear=pygem_prms.gcm_wateryear)
    
# ===== TIME PERIOD =====
dates_table = modelsetup.datesmodelrun(startyear=pygem_prms.gcm_startyear, endyear=pygem_prms.gcm_endyear, spinupyears=pygem_prms.gcm_spinupyears,
            option_wateryear=pygem_prms.gcm_wateryear)
gcm_temp, gcm_dates = gcm.importGCMvarnearestneighbor_xarray(gcm.temp_fn, gcm.temp_vn, main_glac_rgi,
                                                                  dates_table)

TypeError: 'NoneType' object is not iterable

In [21]:
#manually set number of exponentially scaling bins
n_vert_bins = 10
n_points = len(geo_index)
option_bin = 1

#create variable to store glacier geometry
vert_bins = xr.Dataset(data_vars = dict(
    bin_depth = (['pt','vert_idx'],np.zeros((n_points,n_vert_bins))),
    bin_width = (['pt'],np.zeros(n_points)),
    bin_elev = (['pt'],np.zeros(n_points))),
    coords=dict(
        point=(['pt'],geo_index),
        vert_idx=range(n_vert_bins)
        )
    )

#fill vertical bin heights based on ice thickness
for point in geo_index:
    #define surface elvation, width and ice thickness (m) of the current bin
    bin_z, bin_w, bin_h = list(geometry.loc[point])
    if option_bin==1:
        hs = [0.1,.25,.5,.75,1,2,5,10]
        bin_hs = xr.DataArray(hs.append(bin_h-sum(hs)), coords={'pt':point})
        vert_bins['bin_depth'] = bin_hs
    elif option_bin==0:
        c = opt.fsolve(lambda c: bin_h-np.sum(np.exp(np.arange(n_vert_bins)*c)),10)
        bin_hs = xr.DataArray(np.exp(c*range(1,n_vert_bins)), coords={'pt':point})
print(vert_bins)

ValueError: dimension 'loc' already exists as a scalar variable

In [24]:
geopotential = xr.open_dataset(gcm.var_fp+gcm.temp_fn)
# need to (1) process data that I have from Copernicus, aggregate it into new files that are just ERA5-daily-variable.nc that can be accessed easily
# (2) read the ERA5 files to get arrays of geopotential, 
binned_climate = xr.Dataset(
    data_vars = dict(
    bin_geopotential = (['pt','vert_idx'],geopotential),
    bin_width = (['pt'],np.zeros(n_points)),
    bin_elev = (['pt'],np.zeros(n_points))),
    coords=dict(
        point=(['pt'],geo_index),
        vert_idx=range(n_vert_bins)
        )
    )


TypeError: Could not convert tuple of form (dims, data[, attrs, encoding]): (['loc', 'vert_idx'], <xarray.Dataset>
Dimensions:    (longitude: 2, latitude: 2, expver: 2, time: 498)
Coordinates:
  * longitude  (longitude) float32 87.0 87.25
  * latitude   (latitude) float32 28.0 27.75
  * expver     (expver) int32 1 5
  * time       (time) datetime64[ns] 1979-01-01 1979-02-01 ... 2020-06-01
Data variables:
    t2m        (time, expver, latitude, longitude) float32 ...
Attributes:
    Conventions:  CF-1.6
    history:      2020-07-20 17:11:28 GMT by grib_to_netcdf-2.16.0: /opt/ecmw...) to Variable.

In [ ]:
import pygem.pygem_input as pygem_prms
import pickle

In [ ]:
glacier_str = glac_no[0][1::]
modelprms_fn =  glacier_str + '-modelprms_dict.pkl'
modelprms_fp = pygem_prms.output_filepath + 'calibration/' + glacier_str.split('.')[0].zfill(2) + '/'
modelprms_fullfn = modelprms_fp + modelprms_fn
assert os.path.exists(modelprms_fullfn), glacier_str + ' calibrated parameters do not exist.'            
with open(modelprms_fullfn, 'rb') as f:
    modelprms_dict = pickle.load(f)